In [1]:
import os
import pandas as pd
from tqdm import tqdm

In [2]:
path = '/Users/stevie/repos/lingo_kit_combined/lingo_kit_data/word_analysis/datasets/tatoeba/new_token_data/combined/combined_tokens_0_to_560000.tsv'
df = pd.read_csv(path, sep='\t')

In [3]:
len(df), df.columns.tolist()

(44915,
 ['text',
  'lemma',
  'pos',
  'token_count',
  'token_pct',
  'group_count',
  'group_pct',
  'xpos',
  'deprel',
  'feats',
  'token_hash',
  'group_hash',
  'sentences'])

In [4]:
df['group'] = df.apply(lambda row: f"lemma={row['lemma']}, pos={row['pos']}", axis=1)
print(len(df['group'].unique()))
print(len(df['group_hash'].unique()))

23483
23483


In [5]:
sentence_df = pd.read_csv('sentence_dataframe.tsv', sep='\t')
len(sentence_df), sentence_df.columns

(624335, Index(['text_it', 'text_en', 'hash'], dtype='object'))

In [6]:
sentence_df.set_index('hash', inplace=True, drop=True)

In [7]:
group_pct_dict = {}
group_hash_to_str = {}
group_hashes = list(df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    row = df[df['group_hash'] == group_hash].iloc[0]
    group_pct_dict[group_hash] = row['group_pct']
    group_hash_to_str[group_hash] = row['group']

100%|██████████| 23483/23483 [00:27<00:00, 840.54it/s]


In [8]:
all_groups = list(df['group_hash'].unique())
all_groups = sorted(all_groups, key=lambda x: group_pct_dict[x], reverse=True)
[group_hash_to_str[hash] for hash in all_groups[:10]]

['lemma=il, pos=DET',
 'lemma=essere, pos=AUX',
 'lemma=di, pos=ADP',
 'lemma=non, pos=ADV',
 'lemma=a, pos=ADP',
 'lemma=avere, pos=AUX',
 'lemma=uno, pos=DET',
 'lemma=che, pos=SCONJ',
 'lemma=in, pos=ADP',
 'lemma=io, pos=PRON']

In [9]:
N = 200
top_groups = all_groups[:N]
reduced_df = df[df['group_hash'].isin(top_groups)]
len(reduced_df)

2744

In [10]:
reduced_df.columns

Index(['text', 'lemma', 'pos', 'token_count', 'token_pct', 'group_count',
       'group_pct', 'xpos', 'deprel', 'feats', 'token_hash', 'group_hash',
       'sentences', 'group'],
      dtype='object')

In [11]:
save_dir = 'dataframes'

In [ ]:
group_hashes = list(reduced_df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    group_df = reduced_df[reduced_df['group_hash'] == group_hash].copy()
    assert(group_df['lemma'].nunique() == 1)
    assert(group_df['pos'].nunique() == 1)
    pos = group_df.iloc[0]['pos']
    lemma = group_df.iloc[0]['lemma']

    for i, row in group_df.iterrows():
        sentence_hashes = eval(row['sentences'])
        assert(type(sentence_hashes) is list)
        for j in range(3):
            if len(sentence_hashes) > j:
                sentence_hash = sentence_hashes[j]
                sentence_it = sentence_df.loc[sentence_hash, 'text_it']
                sentence_en = sentence_df.loc[sentence_hash, 'text_en']
                group_df.loc[i, f'sentence_{j+1}_it'] = sentence_it
                group_df.loc[i, f'sentence_{j+1}_en'] = sentence_en
    group_df['translation_en'] = ''
    save_path = os.path.join(save_dir, pos, f"{lemma}.tsv")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    group_df.to_csv(save_path, sep='\t', index=False)

  0%|          | 0/200 [00:00<?, ?it/s]/var/folders/qs/n3mb_ckd47n6s9c65b_06xrw0000gn/T/ipykernel_65569/693131261.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group_df[f'sentence_{j+1}_it'] = sentence_it
/var/folders/qs/n3mb_ckd47n6s9c65b_06xrw0000gn/T/ipykernel_65569/693131261.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group_df[f'sentence_{j+1}_en'] = sentence_en
/var/folders/qs/n3mb_ckd47n6s9c65b_06xrw0000gn/T/ipykernel_65569/693131261.py:18: SettingWithCopyWarning: 
A value is trying to